In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 16 · Reasoning Models + RFT / Distillation

Some Member Services questions are genuinely hard — prior-auth logic, benefit-coordination edge cases. This lab routes those to a **reasoning model** via the live `model-router`, then shows the two levers that make a *small* model reason like a big one: **Reinforcement Fine-Tuning (RFT)** and **distillation**. *Pay for reasoning only when the question needs it.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='acme-live-demo')


---
## Step 1 — Let the router pick a reasoning model (LIVE)

Send a trivial prompt and a hard reasoning prompt through `model-router` and inspect `response.model` — the router escalates the hard one to a reasoning-grade model automatically.


In [ ]:
from _advisor import try_build_client
client = try_build_client()
ROUTER = 'model-router'

def route(prompt, max_tokens=400):
    if client is None:
        return '[mock]', '(mock-router)'
    r = client.chat.completions.create(model=ROUTER, max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}])
    return (r.choices[0].message.content or ''), r.model

trivial = 'What are Acme customer service hours?'
hard = ('A member has Gold plan primary and Bronze secondary coverage. '
        'A $4,200 procedure applies; primary covers 80% after a $500 '
        'deductible already met, secondary covers 50% of the remainder '
        'after its own $300 deductible not yet met. Show the member owes.')

a1, m1 = route(trivial, 120)
a2, m2 = route(hard, 600)
print('TRIVIAL  ->', m1)
print('HARD     ->', m2)
print('\nReasoned answer (hard):', a2[:300])
print('\nSame router, different underlying model:', m1 != m2)


---
## Step 2 — Reinforcement Fine-Tuning (RFT)

SFT (Lab 01) imitates examples. **RFT** optimizes against a **grader** that rewards *correct reasoning outcomes* — ideal when you can score a result (right copay? right tier?) but can't hand-write every reasoning path. Here's the job shape (config, not submitted).


In [ ]:
RFT_JOB = {
    'training_file': 'data/acme_reasoning_rft.jsonl',
    'model': 'o4-mini',                 # a fine-tunable reasoning base
    'method': {
        'type': 'reinforcement',
        'reinforcement': {
            'grader': {
                'type': 'string_check',     # or python/score_model grader
                'name': 'copay_exact_match',
                'operation': 'eq',
                'input': '{{sample.output_text}}',
                'reference': '{{item.expected_amount}}',
            },
            'hyperparameters': {'n_epochs': 3, 'reasoning_effort': 'medium'},
        },
    },
}
import json as _json
print('RFT job spec (submit with client.fine_tuning.jobs.create):')
print(_json.dumps(RFT_JOB, indent=2))
print('\nGrader rewards CORRECT ANSWERS, so the model learns to reason, not memorize.')


---
## Step 3 — Distillation: bottle the reasoning into a cheap model

Once a reasoning model (or RFT model) is right, **distill** it: capture its high-quality outputs and SFT a small, cheap model on them. You keep most of the accuracy at a fraction of the cost/latency — exactly what the router then serves by default.


In [ ]:
DISTILL_PIPELINE = '''
# --- distillation pipeline (outline) ---
# 1) TEACHER: run hard prompts through model-router / o4-mini (reasoning).
# 2) CAPTURE: store {prompt -> best answer} as a stored-completions dataset
#    (Azure OpenAI 'stored completions' can auto-collect these).
# 3) STUDENT: SFT gpt-4o-mini on that dataset (reuse Lab 01 flow).
# 4) EVALUATE: Lab 07 - student must hit the teacher's accuracy bar.
# 5) ROUTE: model-router now serves the cheap student for the common case,
#    escalating to the teacher only for the long tail.
'''
print(DISTILL_PIPELINE)


---
## Takeaways

- The **router already escalates** hard questions to reasoning models — you pay for reasoning only when needed (Step 1, live).
- **RFT** optimizes for *correct outcomes* via a grader, not just imitation — perfect for scorable tasks.
- **Distillation** moves that quality into a cheap model so the common case stays fast and inexpensive.
- Together they bend the cost/quality curve the Decision Advisor (Lab 09) optimizes.

*← The Decision Advisor (Lab 09) routes the `needs_reasoning_ft` flag here.*
